# Ejercicio Práctico: Web Scraping de un E-Commerce de Libros (ETL con BeautifulSoup y Pandas)

En este ejercicio vas a construir por tu cuenta un flujo completo de **Extracción, Transformación y Carga (ETL)** utilizando `requests`, `BeautifulSoup` y `pandas` sobre la tienda online de práctica [Books to Scrape](https://books.toscrape.com/).

### Objetivo del ejercicio
Extraer información de múltiples categorías de libros para construir un `DataFrame` limpio con **4 variables clave por libro**:
1. **Título completo** (`str`)
2. **Precio en libras** (`float`, limpiando el símbolo `£`)
3. **Valoración en estrellas** (`str`, extraída de los atributos de clase)
4. **Disponibilidad en Stock** (`str`, limpiando saltos de línea y espacios)

---

## Paso 1: Importación de librerías y configuración

Importa las librerías necesarias (`time`, `requests`, `BeautifulSoup`, `os` y `pandas`) y define el diccionario `headers` con el `User-Agent`.

Setea el máximo de caracteres a 800 en pandas

In [3]:
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup as bs
import numpy as np

# Configuramos Pandas para que muestre textos largos sin cortarlos
pd.set_option('max_colwidth', 800)

# Definimos nuestro 'disfraz' de navegador (User-Agent) para evitar bloqueos 403
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

## Paso 2: Regla del Francotirador (Inspeccionar y probar con 1 solo libro)

Antes de crear ningún bucle, descarga la página principal `https://books.toscrape.com/`, abre el Inspector del navegador (`F12`) y localiza la estructura HTML de un libro:

In [4]:
# 1. Hacemos la petición de prueba a la portada de la tienda
url = "https://books.toscrape.com/"

# Verificamos que responde 200 OK
response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

# 2. Creamos la sopa
soup = bs(response.content, "html.parser")

# 3. Buscamos todas las tarjetas de libros y seleccionamos el primero (índice 0) para probar
book_card = soup.find_all("article", class_="product_pod")

primer_libro = book_card[0]


In [5]:
# Probamos la extracción y limpieza de los 4 datos sobre 'primer_libro'

# 1. TÍTULO
titulo = primer_libro.find("a").get_text()

# 2. PRECIO: Quitamos el símbolo '£' y convertimos a float
precio = primer_libro.find("p", class_="price_color").get_text()
precio = precio.replace("£","")
precio = float(precio)

# 3. ESTRELLAS
estrellas = primer_libro.find("p", class_="star-rating Three").get_attribute_list("class")[-1]


# 4. STOCK: Usamos strip=True en get_text() para limpiar los saltos de línea
stock = primer_libro.find("p",class_="instock availability").get_text(strip=True)

print(f"Título:    {titulo}")
print(f"Precio:    {precio} ({type(precio).__name__})")
print(f"Estrellas: {estrellas}")
print(f"Stock:     {stock}")

Título:    
Precio:    51.77 (float)
Estrellas: Three
Stock:     In stock


## Paso 3: Extracción Multi-Página con Bucle y `time.sleep`

Ahora que sabemos extraer los 4 datos de una tarjeta, vamos a recorrer **3 categorías distintas de la tienda** (*Mystery*, *Science Fiction* y *Travel*) y a extraer los primeros 10 libros de cada una.

* **Regla ética y técnica:** Recuerda añadir `time.sleep(1)` al final de cada página para no saturar el servidor.

In [6]:
# Lista de tuplas con (Nombre_Categoria, URL) a scrapear
urls_categorias = [
    ("Misterio",        "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"),
    ("Ciencia Ficción", "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html"),
    ("Viajes",          "https://books.toscrape.com/catalogue/category/books/travel_2/index.html")
]

# Inicializamos el diccionario vacío donde acumularemos los datos de todas las páginas
datos_libros = {
    "Categoria": [],
    "Titulo": [],
    "Precio_Libras": [],
    "Estrellas": [],
    "Stock": []
}

# BUCLE EXTERIOR: recorre cada categoría
for categoria, url in urls_categorias:

    # Bloque fijo de petición y BeautifulSoup
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup_pagina = bs(response.content, "html.parser")

    libros = soup_pagina.find_all("article", class_="product_pod")

    print(libros[0].find("h3").find("a").get_text())
    # BUCLE INTERIOR: recorre los primeros 10 libros de esa categoría
    for libro in libros[:11]:

        # 1. Título
        libro_titulo = libro.find("h3").find("a").get_text()

        # 2. Precio
        precio = libro.find("p",class_="price_color").get_text()
        precio = precio.replace("£","")
        precio = float(precio)


        # 3. Estrellas
        estrellas = libro.find("p", class_="star-rating").get_attribute_list("class")[-1]

        # 4. Stock
        stock = libro.find("p",class_="instock availability").get_text(strip=True)


        # Guardamos los datos en las listas de nuestro diccionario
        datos_libros["Categoria"].append(categoria)
        datos_libros["Titulo"].append(libro_titulo)
        datos_libros["Precio_Libras"].append(precio)
        datos_libros["Estrellas"].append(estrellas)
        datos_libros["Stock"].append(stock)

    # Esperamos 1 segundo antes de pedir la siguiente página
    time.sleep(1)

print("¡Extracción completada con éxito!")

Sharp Objects
Mesaerion: The Best Science ...
It's Only the Himalayas
¡Extracción completada con éxito!


## Paso 4: Creación del DataFrame, Exploración y Guardado en CSV

1. Convierte el diccionario `datos_libros` en un `DataFrame` de Pandas llamado `df_libros`.
2. Comprueba sus dimensiones (`.shape`) y tipos de datos (`.dtypes`).
3. Calcula el **precio medio por categoría** usando `.groupby()`.
4. Guarda el resultado en `data/libros_scrapeados.csv` con separador `;` y sin índice (`index=False`).

In [7]:
# 1. Creamos el DataFrame a partir del diccionario recolectado
df_libros = pd.DataFrame(datos_libros).set_index("Titulo")

# 2. Comprobamos dimensiones y tipos de datos
print(df_libros.shape)
print(df_libros.dtypes)

# Mostramos los 10 primeros elementos del DataFrame
df_libros.head(10)


(33, 4)
Categoria            str
Precio_Libras    float64
Estrellas            str
Stock                str
dtype: object


,Categoria,Precio_Libras,Estrellas,Stock
Titulo,,,,
Sharp Objects,Misterio,47.82,Four,In stock
"In a Dark, Dark ...",Misterio,19.63,One,In stock
The Past Never Ends,Misterio,56.50,Four,In stock
A Murder in Time,Misterio,16.64,One,In stock
The Murder of Roger ...,Misterio,44.10,Four,In stock
The Last Mile (Amos ...,Misterio,54.21,Two,In stock
That Darkness (Gardiner and ...,Misterio,13.92,One,In stock
Tastes Like Fear (DI ...,Misterio,10.69,One,In stock
A Time of Torment ...,Misterio,48.35,Five,In stock


In [8]:
# 3. Precio medio, mínimo y máximo por categoría usando .groupby()
precios_median = df_libros.groupby("Categoria")["Precio_Libras"].median()
print(precios_median)
print("")

precios_min = df_libros.groupby("Categoria")["Precio_Libras"].min()
print(precios_min)
print("")

precios_max = df_libros.groupby("Categoria")["Precio_Libras"].max()
print(precios_max)


Categoria
Ciencia Ficción    32.42
Misterio           26.80
Viajes             38.95
Name: Precio_Libras, dtype: float64

Categoria
Ciencia Ficción    10.65
Misterio           10.69
Viajes             23.21
Name: Precio_Libras, dtype: float64

Categoria
Ciencia Ficción    48.74
Misterio           56.50
Viajes             56.88
Name: Precio_Libras, dtype: float64


In [9]:
# 4. Guardamos el DataFrame en un archivo CSV limpio usando la librería 'os'

import os
os.makedirs("data", exist_ok=True)

df_libros.to_csv("data/datos_libros.csv",encoding="UTF-8")